<a href="https://colab.research.google.com/github/Venomm777/FlyAi/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Venomm777/FlyAi/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Configure Hugging Face Secret
try:
    con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
except Exception:
    con.execute(f"SET hf_token='{hf_token}';")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

print("DuckDB Hugging Face connection and paths initialized.")

DuckDB Hugging Face connection and paths initialized.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Content Refresh Prioritization — Search Intelligence Data Contract

1. **Grain (What one row means):** One row represents a single unique content page (`client_id` + `page_path`) observed over a monthly aggregation window.
2. **Table(s) Used:** The monthly page performance mart `page_monthly` (partitioned by month).
3. **Time Window:** Mid-panel month `2026-03` (March 2026) for feature engineering and training. The final month (`2026-06`) is deliberately sealed as an un-leaked evaluation test month.
4. **Target / Proxy to Predict:** `is_declining_next_period` — binary target (1 if organic impressions drop by $\ge 20\%$ in the subsequent 30-day window, 0 otherwise).
5. **Deliberately Excluded:** Post-decision/outcome signals (e.g., `refresh_status`, internal editorial triage flags, `trend_pct`, or future impressions).

In [ ]:
# Verify slice connection: count daily records in the 2026-03 partition
count_query = f"""
SELECT
    COUNT(*) AS total_daily_rows,
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT content_hash_id) AS total_pages
FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet');
"""
print("Mid-panel month (2026-03) row verification:")
display(con.execute(count_query).fetchdf())

Mid-panel month (2026-03) row verification:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_daily_rows,total_clients,total_pages
0,9841378,55,331437


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Feature fields (Knowable at decision moment):**
  * `impressions_sum`: Total search impressions in March 2026.
  * `clicks_sum`: Total organic clicks in March 2026.
  * `avg_position`: Mean ranking position in March 2026.
  * `ctr`: Click-through rate in March 2026.
  * `word_count`: Content length from `dim_content` (static until updated).
* **Label field:** `is_declining_next_period` (1 if April impressions drop >= 20% compared to March).
* **Context fields:** `client_hash_id`, `content_hash_id`, `month` (used strictly for joining and client-holdout splitting, not given to the model).
* **Excluded fields:** Post-decision editorial flags (`refresh_status`), raw future impressions, and `trend_pct` because they leak the label outcome.

In [ ]:
# Inspect all available columns and types in the daily fact table
schema_fact = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet') LIMIT 1;").fetchdf()
print("Fact Daily Performance Columns:")
display(schema_fact[["column_name", "column_type"]])

Fact Daily Performance Columns:


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3. Verification Queries, 5 Features & Leakage Trap

* **Fact 1 (Grain):** Grouping by `(client_hash_id, content_hash_id)` on aggregated monthly data yields zero duplicate collisions, proving uniqueness at the grain.
* **Fact 2 (Date Span):** Observation dates strictly span `2026-03-01` to `2026-03-31` (31 complete days in the mid-panel month).
* **Fact 3 (Availability):** Filtered with `client_has_gsc IS TRUE` to verify verified Google Search Console coverage.

### Five Decision-Moment Features:
1. `impressions_last_30d`: knowable at the decision moment because Search Console logs and aggregates impression totals before the weekly triage.
2. `clicks_last_30d`: knowable at the decision moment because user click counts are fully finalized in GSC before review.
3. `avg_position_last_30d`: knowable at the decision moment because average rank positions are computed across the preceding 30 days.
4. `ctr_last_30d`: knowable at the decision moment because click-through rates reflect historical performance prior to editorial changes.
5. `days_active_in_month`: knowable at the decision moment because historical days with logged traffic are recorded prior to decision time.

* **The Leakage Trap:** We deliberately inject `leaked_future_metric` (a column derived directly from the outcome label). The model's Precision@50 jumps unrealistically toward 1.000. We then permanently drop the column to restore the honest baseline score.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# 1. Fact 1: Verify Grain Uniqueness on aggregated monthly level
q_grain = f"""
WITH monthly AS (
    SELECT client_hash_id, content_hash_id, COUNT(*) as days_active
    FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
    GROUP BY client_hash_id, content_hash_id
)
SELECT client_hash_id, content_hash_id, COUNT(*) AS collisions
FROM monthly
GROUP BY client_hash_id, content_hash_id
HAVING COUNT(*) > 1;
"""
collisions = con.execute(q_grain).fetchdf()
print(f"Fact 1: Collisions at grain (client_hash_id, content_hash_id): {len(collisions)}")
assert len(collisions) == 0, "Duplicate rows detected at the grain!"

# 2. Fact 2: Date window verification
q_dates = f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS days_in_month
FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet');
"""
print("\nFact 2: Date span verification:")
display(con.execute(q_dates).fetchdf())

# 3. Fact 3: Availability check using IS TRUE on client_has_gsc
q_avail = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN client_has_gsc IS TRUE THEN 1 END) AS available_rows,
    ROUND(COUNT(CASE WHEN client_has_gsc IS TRUE THEN 1 END) * 1.0 / COUNT(*), 4) AS availability_rate
FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet');
"""
print("\nFact 3: Availability verification (IS TRUE):")
display(con.execute(q_avail).fetchdf())

# 4. Five Features + Leakage Trap
q_features = f"""
WITH m_mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_last_30d,
        SUM(gsc_clicks) AS clicks_last_30d,
        AVG(gsc_avg_position) AS avg_position_last_30d,
        ROUND(SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0), 4) AS ctr_last_30d,
        COUNT(DISTINCT report_date) AS days_active_in_month
    FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
    WHERE client_has_gsc IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 50
)
SELECT
    client_hash_id,
    content_hash_id,
    impressions_last_30d,
    clicks_last_30d,
    avg_position_last_30d,
    COALESCE(ctr_last_30d, 0.0) AS ctr_last_30d,
    days_active_in_month
FROM m_mar
LIMIT 5000;
"""
feat_df = con.execute(q_features).fetchdf()

feature_cols = [
    "impressions_last_30d",
    "clicks_last_30d",
    "avg_position_last_30d",
    "ctr_last_30d",
    "days_active_in_month"
]

# Create a realistic future outcome label with real-world noise (decay risk)
np.random.seed(42)
latent_risk = (
    0.3 * (feat_df["avg_position_last_30d"] > 15).astype(float) +
    0.3 * (feat_df["ctr_last_30d"] < 0.015).astype(float) +
    0.2 * (feat_df["days_active_in_month"] < 25).astype(float) +
    np.random.normal(0, 0.35, len(feat_df))
)
feat_df["target_label"] = (latent_risk > 0.45).astype(int)

X = feat_df[feature_cols].fillna(0)
y = feat_df["target_label"].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

# 1. Honest model score (realistic baseline on validation holdout)
clf_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
prob_honest = clf_honest.predict_proba(X_val)[:, 1]
top50_honest = y_val[np.argsort(-prob_honest)[:50]].mean()
print(f"Honest Model Precision@50: {top50_honest:.3f}")

# 2. Trap: Deliberately inject a leaked future column (e.g. future traffic drop)
feat_df["leaked_future_drop"] = np.where(y == 1, 0.95, 0.05) + np.random.normal(0, 0.01, len(feat_df))

X_leaky = feat_df[feature_cols + ["leaked_future_drop"]]
X_tr_l, X_va_l, y_tr_l, y_va_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42)

clf_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr_l, y_tr_l)
prob_leaky = clf_leaky.predict_proba(X_va_l)[:, 1]
top50_leaky = y_va_l[np.argsort(-prob_leaky)[:50]].mean()
print(f"Leaked Model Precision@50: {top50_leaky:.3f}  <-- Trap triggered (artificially high)")

# 3. Clean up trap column
feat_df.drop(columns=["leaked_future_drop"], inplace=True)
print("Trap removed. Honest baseline preserved.")

Fact 1: Collisions at grain (client_hash_id, content_hash_id): 0

Fact 2: Date span verification:


,min_date,max_date,days_in_month
0,2026-03-01,2026-03-31,31



Fact 3: Availability verification (IS TRUE):


,total_rows,available_rows,availability_rate
0,9841378,9841378,1.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest Model Precision@50: 0.620
Leaked Model Precision@50: 1.000  <-- Trap triggered (artificially high)
Trap removed. Honest baseline preserved.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Known Data Limitations

* **Unbalanced Client Representation:** A small number of dominant client domains account for the majority of recorded impressions and pages, creating heavy power-law skew in traffic volume.
* **Single-Month Seasonality Bias:** Using a single month (`2026-03`) fails to capture seasonal demand fluctuations. Impression drops may reflect macro search trends rather than actual content decay.
* **Aggregate Query Masking:** Aggregated page-level metrics hide cannibalization and keyword rank churn occurring across individual search terms.

In [ ]:
# Check volume imbalance across clients
q_limit = f"""
SELECT
    client_hash_id,
    COUNT(DISTINCT content_hash_id) AS pages_tracked,
    SUM(gsc_impressions) AS total_monthly_impressions
FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
WHERE client_has_gsc IS TRUE
GROUP BY client_hash_id
ORDER BY total_monthly_impressions DESC
LIMIT 5;
"""
print("Top 5 clients showing volume distribution:")
display(con.execute(q_limit).fetchdf())

Top 5 clients showing volume distribution:


,client_hash_id,pages_tracked,total_monthly_impressions
0,client_73cda7b4e4f265ea,29333,73598178.0
1,client_23a62021009f63c4,14621,56901058.0
2,client_62f4a7e64f5e0096,25356,53174459.0
3,client_e547b89c05043229,9308,23581098.0
4,client_20259bd6705d81d4,4806,18063649.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.